In [ ]:
import pandas as pd
import numpy as np

train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")
test_woe = pd.read_csv("../data/processed/test_woe.csv")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Load datasets
train_woe = pd.read_csv("../data/processed/train_woe.csv")
validation_woe = pd.read_csv("../data/processed/validation_woe.csv")

target = "SeriousDlqin2yrs"

final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

# Fit model again
X_train = sm.add_constant(train_woe[final_variables])
y_train = train_woe[target]

model = sm.Logit(y_train, X_train).fit()

# Predict PD on validation
X_validation = sm.add_constant(validation_woe[final_variables])
validation_woe["pd"] = model.predict(X_validation)

# Scorecard scaling parameters
PDO = 20
BASE_SCORE = 600
BASE_ODDS = 20

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

# Calculate logit and score
validation_woe["logit"] = np.dot(
    X_validation,
    model.params
)

validation_woe["score"] = (
    offset - factor * validation_woe["logit"]
)

validation_woe[["pd", "score", target]].head()

## Decile Analysis

In [ ]:
validation_woe["score_decile"] = pd.qcut(
    validation_woe["score"],
    q=10,
    labels=False
)

In [ ]:
decile_analysis = (
    validation_woe
    .groupby("score_decile")
    .agg(
        customers=("score","count"),
        avg_score=("score","mean"),
        avg_pd=("pd","mean"),
        bad_rate=("SeriousDlqin2yrs","mean")
    )
    .reset_index()
)

decile_analysis

## Approval Rate Function

In [ ]:
def approval_analysis(df, cutoff_score):

    approved = df[df["score"] >= cutoff_score]

    approval_rate = (
        len(approved)
        /
        len(df)
    )

    bad_rate = approved[
        "SeriousDlqin2yrs"
    ].mean()

    return approval_rate, bad_rate

In [ ]:
cutoffs = [
    540,
    560,
    580,
    600,
    620,
    640
]

results = []

for cutoff in cutoffs:

    approval_rate, bad_rate = approval_analysis(
        validation_woe,
        cutoff
    )

    results.append(
        {
            "Cutoff": cutoff,
            "Approval Rate": approval_rate,
            "Portfolio Bad Rate": bad_rate
        }
    )

cutoff_analysis = pd.DataFrame(results)

cutoff_analysis

In [ ]:
cutoff_analysis["Risk Reduction"] = (
    validation_woe["SeriousDlqin2yrs"].mean()
    - cutoff_analysis["Portfolio Bad Rate"]
)

## Cut-Off Determination Summary

The scorecard demonstrates strong rank ordering across deciles. The lowest score decile has a bad rate of 31.0%, while the highest score decile has a bad rate of 0.8%.

Different score cut-offs were evaluated to assess the trade-off between approval rate and accepted portfolio bad rate.

A potential operational strategy is:

- Score < 580: Reject
- Score 580–600: Manual Review
- Score >= 600: Approve

At a cut-off of 600, the approval rate is approximately 65.6% and the approved portfolio bad rate is approximately 1.9%.

The final cut-off should be selected based on risk appetite, profitability, operational capacity for manual review and business growth objectives.